### Imports

In [ ]:
import sys
import glob
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d

# Make sure Python can find plot_era40.py in the parent directory
sys.path.append("..")
from plot_era40 import _valid_time, _reduced_to_full   # importing plot_era40 also sets up eccodes in the right order
import eccodes as ec

### Central latitudes from MHW (hardcoded, every 5° of longitude)

*From the paper's figure (values read off the image, so approximate).*

In [ ]:
mfw_hardcoded_lats = [56.5, 56.6, 57.1, 57.8, 58.2, 58.8, 58.9, 59.1,
                    60.0, 60.0, 60.0, 60.0, 60.0, 59.6, 59.0, 58.9, 58.9, 58.9,
                    58.8, 57.8, 57.2, 55.3, 52.4, 49.1, 46.8, 45.2, 44.4, 44.3,
                    44.3, 44.3, 44.4, 44.7, 45.4, 45.4, 45.4, 45.4, 45.4, 45.5,
                    45.5, 45.6, 46.1, 46.4, 46.6, 47.4, 47.6, 47.6, 47.7, 47.8,
                    48.5, 48.8, 48.8, 48.7, 47.8, 47.7, 46.8, 46.7, 47.4, 47.6,
                    47.8, 48.1, 48.8, 49.7, 50.5, 51.3, 52.1, 52.8, 53.4, 54.2,
                    54.4, 55.2, 55.5, 55.5]

mfw_hardcoded_lats = np.array(mfw_hardcoded_lats)


### Band-pass (2–6 day) eddy filter for $\phi_c$

The central latitude is the latitude of maximum variance of the 2–6 day band-passed wind (TH08: "synoptic time-scale transient eddy kinetic energy").

In [ ]:
from scipy.signal import butter, sosfiltfilt

def synoptic_eke(u_timeseries, v_timeseries):
    """
    Transient eddy kinetic energy at each time step, keeping only 2-6 day (synoptic) fluctuations.

    Parameters:
    - u_timeseries, v_timeseries: 3D numpy arrays of shape (time, lat, lon), 6-hourly,
      one continuous stretch of time (no gaps)

    Returns:
    - eke: 3D numpy array of shape (time, lat, lon)
    """
    # 4th-order Butterworth band-pass; fs = 4 samples per day, band = 1/6 to 1/2 cycles per day
    sos = butter(4, [1/6, 1/2], btype="bandpass", fs=4, output="sos")
    # sosfiltfilt runs the filter forwards and backwards, so it does not shift events in time
    u_prime = sosfiltfilt(sos, u_timeseries, axis=0)
    v_prime = sosfiltfilt(sos, v_timeseries, axis=0)
    return (0.5 * (u_prime**2 + v_prime**2)).astype(np.float32)

def central_latitudes_from_eke(lats, lons, mean_eke):
    """
    Same as step 3 of compute_central_latitudes, but starting from an already time-averaged EKE.

    Parameters:
    - mean_eke: 2D numpy array of shape (len(lats), len(lons))

    Returns:
    - central_lats: dictionary mapping each longitude value to its central latitude
    """
    central_lats = {}
    for lon_idx, lon in enumerate(lons):
        smoothed_eke = uniform_filter1d(mean_eke[:, lon_idx], size=5, mode='reflect')
        valid_mask = (lats >= 20) & (lats <= 80)
        search_eke = np.where(valid_mask, smoothed_eke, -np.inf)
        central_lats[lon] = lats[np.argmax(search_eke)]
    return central_lats


### Read u and v from the full ERA-40 files

`../data/e4oper.an.pv.YYYYMM` (not in git). These hold the wind on the 2 PVU surface; the paper used 300 hPa, which is not in our data for every month.

In [ ]:
def read_uv_month(path):
    """Return lats, lons, u, v with shape (time, lat, lon) from one e4oper.an.pv.YYYYMM file, in time order."""
    lats = lons = None
    fields = {"u": [], "v": []}
    with open(path, "rb") as f:
        while (gid := ec.codes_grib_new_from_file(f)) is not None:
            try:
                name = ec.codes_get(gid, "shortName")
                if name not in fields:
                    continue
                if ec.codes_get(gid, "gridType") == "reduced_gg":
                    arr = _reduced_to_full(ec.codes_get_values(gid), ec.codes_get_array(gid, "pl").astype(int))
                else:
                    arr = ec.codes_get_values(gid).reshape(ec.codes_get(gid, "Nj"), ec.codes_get(gid, "Ni"))
                if lats is None:
                    lats = ec.codes_get_array(gid, "distinctLatitudes")
                    lons = np.arange(arr.shape[1]) * 360.0 / arr.shape[1]
                fields[name].append((_valid_time(gid), arr.astype(np.float32)))
            finally:
                ec.codes_release(gid)
    u, v = (np.array([a for _, a in sorted(fields[n], key=lambda p: p[0])]) for n in ("u", "v"))
    return lats, lons, u, v

### Annual central latitude, 1979–1993

The paper's curve uses ERA-15, i.e. 1979–1993, all months. Each year is filtered as one continuous series.

In [ ]:
YEARS = range(1979, 1994)

eke_sum, n_steps = 0, 0
for year in YEARS:
    months = [read_uv_month(f"../data/e4oper.an.pv.{year}{m:02d}") for m in range(1, 13)]
    lats, lons = months[0][0], months[0][1]
    eke = synoptic_eke(np.concatenate([m[2] for m in months]), np.concatenate([m[3] for m in months]))
    eke_sum = eke_sum + eke.sum(axis=0)
    n_steps += len(eke)
    print(year, "done")

cbl_annual = central_latitudes_from_eke(lats, lons, eke_sum / n_steps)   # longitude -> latitude of the maximum

### Smooth in longitude and test against MHW's curve

The paper's curve is "a smoothed version"; the smoothing isn't specified, so a 45° running mean is used. TH08 report seasonal departures from it of "generally 4° or less", used here as the tolerance.

In [ ]:
SMOOTH_DEG = 45

raw = np.array(list(cbl_annual.values()))
cbl_smooth = uniform_filter1d(raw, size=int(round(SMOOTH_DEG / (lons[1] - lons[0]))), mode="wrap")

paper_lons = np.arange(0, 360, 5)
diff = np.interp(paper_lons, lons, cbl_smooth, period=360) - mfw_hardcoded_lats
print(f"RMS difference: {np.sqrt(np.mean(diff**2)):.1f}°; largest: {np.abs(diff).max():.1f}° at {paper_lons[np.abs(diff).argmax()]}°E")
print("Within 4° at every longitude:", bool(np.abs(diff).max() <= 4))

plt.figure(figsize=(12, 5))
plt.plot(paper_lons, mfw_hardcoded_lats, color="black", linewidth=2.5, label="MHW (hardcoded)")
plt.plot(lons, raw, color="gray", linewidth=1, label="ours, unsmoothed")
plt.plot(lons, cbl_smooth, color="tab:blue", linewidth=2, label=f"ours, {SMOOTH_DEG}° running mean")
plt.xlim(0, 360)
plt.ylim(20, 90)
plt.xticks(range(0, 361, 60))
plt.xlabel("Longitude (°E)")
plt.ylabel("CBL (°N)")
plt.title("Central blocking latitude: ERA-40 1979–1993 vs MHW")
plt.grid(alpha=0.3)
plt.legend()
plt.show()